In [1]:
import yfinance as yf
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
# from sklearn.model_selection import GridSearchCV
import numpy as np
import os

In [9]:
def load_data():
    if os.path.exists("sp500.csv"):
        sp500 = pd.read_csv("../data/sp500.csv", index_col=0)
    else:
        sp500 = yf.Ticker("^GSPC")
        sp500 = sp500.history(period="max")
        sp500.to_csv("sp500.csv")
    sp500.index = pd.to_datetime(sp500.index)
    return sp500

In [3]:
# Preprocess data
def preprocess_data(sp500):
    sp500 = sp500.copy()
    sp500.drop(columns=["Dividends", "Stock Splits"], inplace=True)
    sp500["Tomorrow"] = sp500["Close"].shift(-1)
    sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)
    sp500 = sp500.loc["1990-01-01":].copy()
    return sp500

In [4]:
# Feature engineering
def add_features(sp500):
    horizons = [2, 5, 60, 250, 1000]
    for horizon in horizons:
        rolling_avg = sp500["Close"].rolling(horizon).mean()
        sp500[f"Close_Ratio_{horizon}"] = sp500["Close"] / rolling_avg
        sp500[f"Trend_{horizon}"] = sp500["Target"].shift(1).rolling(horizon).sum()
    sp500["Volatility_5"] = sp500["Close"].rolling(5).std()
    sp500["Volatility_20"] = sp500["Close"].rolling(20).std()
    sp500.dropna(inplace=True)
    return sp500

In [5]:
# Backtesting framework
def predict(train, test, predictors, model: RandomForestClassifier):
    model.fit(train[predictors], train["Target"])
    probs = model.predict_proba(test[predictors])[:, 1]
    preds = (probs >= 0.6).astype(int)
    predictions = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], predictions], axis=1)
    return combined

In [6]:
def backtest(data, model: RandomForestClassifier, predictors, start=2500, step=250):
    all_predictions = []
    for i in range(start, data.shape[0], step):
        train = data.iloc[:i].copy()
        test = data.iloc[i:(i + step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)
    return pd.concat(all_predictions)

In [7]:
def evaluate(predictions):
    precision = precision_score(predictions["Target"], predictions["Predictions"], zero_division=0)
    recall = recall_score(predictions["Target"], predictions["Predictions"], zero_division=0)
    f1 = f1_score(predictions["Target"], predictions["Predictions"], zero_division=0)
    accuracy = np.mean(predictions["Target"] == predictions["Predictions"])
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")
    print(f"Accuracy: {accuracy:.2f}")
    return precision, recall, f1, accuracy

In [8]:
# Load and preprocess data
sp500 = load_data()
sp500 = preprocess_data(sp500)
sp500 = add_features(sp500)

sp500

,Open,High,Low,Close,Volume,Tomorrow,Target,Close_Ratio_2,Trend_2,Close_Ratio_5,Trend_5,Close_Ratio_60,Trend_60,Close_Ratio_250,Trend_250,Close_Ratio_1000,Trend_1000,Volatility_5,Volatility_20
Date,,,,,,,,,,,,,,,,,,,
1993-12-14 00:00:00-05:00,465.730011,466.119995,462.459991,463.059998,275050000,461.839996,0,0.997157,1.0,0.996617,1.0,1.000283,32.0,1.028047,127.0,1.176082,512.0,1.328341,2.077816
1993-12-15 00:00:00-05:00,463.059998,463.690002,461.839996,461.839996,331770000,463.339996,1,0.998681,0.0,0.995899,1.0,0.997329,32.0,1.025151,126.0,1.172676,512.0,1.426862,1.982740
1993-12-16 00:00:00-05:00,461.859985,463.980011,461.859985,463.339996,284620000,466.380005,1,1.001621,1.0,0.999495,2.0,1.000311,32.0,1.028274,127.0,1.176163,513.0,1.411770,1.955521
1993-12-17 00:00:00-05:00,463.339996,466.380005,463.339996,466.380005,363750000,465.850006,0,1.003270,2.0,1.004991,3.0,1.006561,32.0,1.034781,128.0,1.183537,514.0,1.905178,2.069950
1993-12-20 00:00:00-05:00,466.380005,466.899994,465.529999,465.850006,255900000,465.299988,0,0.999431,1.0,1.003784,2.0,1.005120,32.0,1.033359,128.0,1.181856,513.0,1.938272,2.123810
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-21 00:00:00-05:00,5940.580078,5963.319824,5887.259766,5948.709961,4230120000,5969.339844,1,1.002663,2.0,1.006651,4.0,1.033128,35.0,1.123584,146.0,1.332040,529.0,29.219235,97.144262
2024-11-22 00:00:00-05:00,5944.359863,5972.899902,5944.359863,5969.339844,4141420000,5987.370117,1,1.001731,2.0,1.006778,5.0,1.035579,36.0,1.126273,147.0,1.335971,529.0,29.804624,97.673234
2024-11-25 00:00:00-05:00,5992.279785,6020.750000,5963.910156,5987.370117,5633150000,6021.629883,1,1.001508,2.0,1.006636,5.0,1.037690,36.0,1.128455,147.0,1.339311,530.0,31.314107,98.959191


In [10]:
# Define model and predictors
predictors = ["Close", "Volume", "Open", "High", "Low"] + \
             [f"Close_Ratio_{h}" for h in [2, 5, 60, 250, 1000]] + \
             [f"Trend_{h}" for h in [2, 5, 60, 250, 1000]] + \
             ["Volatility_5", "Volatility_20"]
predictors

['Close',
 'Volume',
 'Open',
 'High',
 'Low',
 'Close_Ratio_2',
 'Close_Ratio_5',
 'Close_Ratio_60',
 'Close_Ratio_250',
 'Close_Ratio_1000',
 'Trend_2',
 'Trend_5',
 'Trend_60',
 'Trend_250',
 'Trend_1000',
 'Volatility_5',
 'Volatility_20']

In [11]:
model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=1)

# Backtest and evaluate
predictions = backtest(sp500, model, predictors)
predictions

,Target,Predictions
Date,,
2003-11-14 00:00:00-05:00,0,0
2003-11-17 00:00:00-05:00,0,0
2003-11-18 00:00:00-05:00,1,1
2003-11-19 00:00:00-05:00,0,0
2003-11-20 00:00:00-05:00,1,1
...,...,...
2024-11-21 00:00:00-05:00,1,0
2024-11-22 00:00:00-05:00,1,0
2024-11-25 00:00:00-05:00,1,0


In [12]:
eval_res = evaluate(predictions)
eval_res

Precision: 0.55
Recall: 0.10
F1 Score: 0.17
Accuracy: 0.46


(0.5531496062992126,
 0.09719820131442407,
 0.16534274786701972,
 0.4642115203021719)